<a href="https://colab.research.google.com/github/IT24104155/se3082-lab-09-2026/blob/main/notebooks/IT24104155_Lab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!nvidia-smi


In [ ]:

!nvcc --version


In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv

In [ ]:
%%writefile hello.cu
#include <cstdio>
__global__ void hello() {
 printf("Hello from GPU thread %d\n", threadIdx.x);
}
int main(void) {
 hello<<<1, 4>>>(); // 1 block of 4 threads
 cudaDeviceSynchronize(); // wait so the GPU's output is flushed
 return 0;
}


In [ ]:

!nvcc -arch=sm_75 hello.cu -o hello && ./hello


Exercise 2: Simple calculation program

In [ ]:

%%writefile ex2_add.cu
#include <cstdio>

__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;
}

int main(void) {
    int a, b, c;
    int *d_a, *d_b, *d_c;
    int size = sizeof(int);

    // Allocate memory on the GPU
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Input values
    a = 2;
    b = 7;

    // Copy inputs from CPU to GPU
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // Launch one block containing one thread
    add<<<1, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));

    // Copy result from GPU to CPU
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);
    printf("Result is %d\n", c);

    // Release GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:

!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add


Exercise 3: Vector addition using 512 blocks and one thread

In [ ]:

%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // Each block handles one element
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and initialize inputs
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 512 blocks, 1 thread per block
    add<<<N, 1>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));

    // Copy result back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print all results
    for (int r = 0; r < N; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Verify results
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    // Cleanup
    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

Exercise 4: Vector addition using one block and 512 threads

In [ ]:

%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // Host copies of a, b, c
    int *d_a, *d_b, *d_c;       // Device copies of a, b, c
    int size = N * sizeof(int);

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies and set up input values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Launch addT() with 1 block of N threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print all results
    for (int r = 0; r < N; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

Exercise 5: Multiply two vectors using blocks and threads

In [ ]:

%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: Calculate the global index of this thread
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: Compute only if the index is inside the array
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // Host copies of a, b, c
    int *d_a, *d_b, *d_c;       // Device copies of a, b, c
    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate input values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: Calculate the number of blocks, rounding up
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    // TODO 4: Launch the kernel
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));

    // TODO 5: Copy the result back to the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 6: Print the last 1000 results
    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n",
               i, a[i], b[i], c[i]);
    }

    // TODO 7: Verify all N results on the CPU
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Exercise 6 – Multiply two 10,000 × 10,000 matrices element by element


In [ ]:

%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>

#define ROWS 10000
#define COLS 10000

// Element-wise product: C[r][c] = A[r][c] * B[r][c]
__global__ void mulElem(int *a, int *b, int *c,
                        int rows, int cols) {
    // TODO 1: Calculate the row and column of this thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: Check that both coordinates are within bounds
    if (row < rows && col < cols) {
        // TODO 3: Convert the 2D coordinates to a flat index
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    int *a, *b, *c;             // Host copies, stored as flat arrays
    int *d_a, *d_b, *d_c;       // Device copies

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate input values
    a = (int *)malloc(size);
    random_ints(a, count);

    b = (int *)malloc(size);
    random_ints(b, count);

    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);

    // TODO 4: Calculate the grid dimensions, rounding up
    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks\n",
           numBlocks.x, numBlocks.y);

    // TODO 5: Launch the kernel
    mulElem<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, ROWS, COLS
    );

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));

    // TODO 6: Copy the result back to the host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: Print C[9999][9000] through C[9999][9999]
    printf("\nLast 1000 matrix elements:\n");

    for (int col = 9000; col < COLS; col++) {
        int idx = (ROWS - 1) * COLS + col;

        printf("C[%d][%d] = %d x %d = %d\n",
               ROWS - 1, col, a[idx], b[idx], c[idx]);
    }

    // TODO 8: Verify all elements using the CPU
    int errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i])
            errors++;
    }

    printf("\nVerification: %d mismatches out of %zu\n",
           errors, count);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix